# 📦 Sandbox Compute with VideoDB

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/sandbox/sandbox_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Use VideoDB Sandbox as dedicated compute for self-hosted indexing and generative AI workloads.

## 🧭 Flow

1. Create a sandbox and wait until it is ready.
2. Run scene indexing on sandbox compute.
3. Generate OmniVoice audio, including a reusable voice clone.
4. Generate FLUX images.
5. Combine generated image and narration into a playable stream.
6. Stop the sandbox when done to end compute billing.

> If `sandbox_id` is omitted for some self-inference models, VideoDB can auto-pick a compatible active sandbox. Passing `sandbox_id` explicitly is recommended for predictable routing.

## 🛠️ Setup

Install the SDK and connect to VideoDB.

In [ ]:
!pip install -q -U videodb python-dotenv

In [ ]:
import os
from getpass import getpass

from videodb import connect, SandboxTier, IndexType, SearchType, SceneExtractionType, play_stream
from videodb.editor import Timeline, Track, Clip, ImageAsset, AudioAsset, Fit

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect()
coll = conn.get_collection()
print(f"✅ Connected to VideoDB collection: {coll.id}")

## 🤖 Sandbox models and tiers

Choose the smallest tier that supports the largest model in your workflow.

| Model | Type | Minimum tier |
|---|---|---|
| `google/gemma-4-E2B-it-FP8` | VLM | `small` |
| `google/gemma-4-12B-it-FP8` | VLM | `medium` |
| `google/gemma-4-26B-A4B-it-FP8` | VLM | `medium` |
| `google/gemma-4-31B-it-FP8` | VLM | `medium` |
| `Qwen/Qwen3.5-9B-FP8` | VLM | `small` |
| `Qwen/Qwen3.5-27B-FP8` | VLM | `medium` |
| `openai/whisper-large-v3-turbo` | Speech-to-text | `small` |
| `k2-fsa/OmniVoice` | Text-to-speech | `small` |
| `black-forest-labs/FLUX.1-dev` | Image generation | `medium` |
| `stabilityai/stable-audio-open-1.0` | Audio generation | `small` |
| `rtdetr-v2-r50vd` | Object detection | `small` |

This notebook creates a `medium` sandbox so all examples can run on the same sandbox.

In [ ]:
VLM_MODEL = "google/gemma-4-31B-it-FP8"
TTS_MODEL = "k2-fsa/OmniVoice"
IMAGE_MODEL = "black-forest-labs/FLUX.1-dev"

## 🏗️ 1. Create a sandbox

A sandbox is a warm compute pool for model workloads. Creation returns immediately, usually in a `provisioning` state.

In [ ]:
sandbox = conn.create_sandbox(tier=SandboxTier.medium)
print(f"Sandbox: {sandbox.id}, status: {sandbox.status}, tier: {sandbox.tier}")

In [ ]:
sandbox.wait_for_ready(timeout=300, interval=5)
print(f"✅ Sandbox ready: {sandbox.id}, status: {sandbox.status}")

In [ ]:
# Optional: inspect available sandboxes
for sb in conn.list_sandboxes():
    print(f"{sb.id} | {sb.name} | {sb.tier} | {sb.status}")

## 🎬 2. Upload a video and run scene indexing

Pass `sandbox_id=sandbox.id` to route scene indexing to your dedicated compute pool.

In [ ]:
video = coll.upload("https://www.youtube.com/watch?v=jeA-KBv0b68")
print(f"Video ID: {video.id}")

In [ ]:
index_id = video.index_scenes(
    extraction_type=SceneExtractionType.time_based,
    extraction_config={
        "time": 10,
        "select_frames": ["first"],
        "frame_count": 1,
    },
    model_name=VLM_MODEL,
    prompt="Describe the scene in a clear, concise way.",
    sandbox_id=sandbox.id,
)
print(f"Scene index ID: {index_id}")

In [ ]:
# Re-run this cell until the index is ready.
idx = video.get_scene_index(index_id)
print(idx)
print(f"Indexed scenes: {len(idx) if idx else 0}")

In [ ]:
query = "Claude"
res = video.search(query, index_type=IndexType.scene, search_type=SearchType.semantic)
shots = res.get_shots()

print(f"Found {len(shots)} result(s) for '{query}':")
for i, shot in enumerate(shots, start=1):
    print(f"{i}. {shot.start:.2f}s - {shot.end:.2f}s")

stream_url = res.compile()
print(f"Stream: {stream_url}")
play_stream(stream_url)

## 🎙️ 3. OmniVoice text-to-speech

Use `coll.generate_voice(..., model_name=TTS_MODEL, sandbox_id=sandbox.id)`. The examples below show basic TTS, voice design, reusable voice clone, and language/speed config.

In [ ]:
def demo_audio(audio_id):
    audio = coll.get_audio(audio_id)

    timeline = Timeline(conn)
    timeline.resolution = "1280x720"
    timeline.background = "#000000"

    audio_track = Track()
    audio_track.add_clip(0, Clip(asset=AudioAsset(id=audio_id), duration=float(audio.length)))
    timeline.add_track(audio_track)

    stream_url = timeline.generate_stream()
    player_url = f"https://player.videodb.io/watch?v={stream_url}"
    print(f"Stream: {stream_url}")
    print(f"Player: {player_url}")
    return player_url

### 🗣️ Basic TTS

In [ ]:
job = coll.generate_voice(
    text="Welcome builders. With VideoDB Sandbox, your indexing and generation jobs run on dedicated compute.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)

### 🎚️ Voice design

In [ ]:
job = coll.generate_voice(
    text="Product update: teams can now launch dedicated inference compute for video understanding and generative media workflows.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
    config={"instructions": "A deep, authoritative male news anchor voice"},
)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)

### 🧬 Reusable voice clone

Create a `VoiceClone` once from a reference audio asset, then pass `voice_clone_id` to future OmniVoice generations.

In [ ]:
ref_audio = coll.upload(
    url="https://www.youtube.com/shorts/8GrguhmR6oQ",
    media_type="audio",
)
print(f"Reference audio ID: {ref_audio.id}, length: {ref_audio.length}s")

voice_clone = coll.create_voice_clone(
    ref_audio_id=ref_audio.id,
    name="Product Narrator",
    description="Reusable OmniVoice clone for product demos",
    ref_text="Sample reference text for the audio clip",
    language="en",
)
print(f"Voice clone ID: {voice_clone.id}")

job = coll.generate_voice(
    text="This is your reusable product narrator. Generate consistent voiceovers for demos, explainers, and customer updates.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
    voice_clone_id=voice_clone.id,
)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)

### ⚙️ Additional TTS config

In [ ]:
job = coll.generate_voice(
    text="Hola. Con VideoDB Sandbox, tus flujos de video pueden usar cómputo dedicado para inferencia.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
    config={"speed": 1.2, "language": "es"},
)

audio = job.wait(timeout=900, interval=5)
print(audio)

## 🖼️ 4. FLUX image generation

Use `coll.generate_image(..., model_name=IMAGE_MODEL, sandbox_id=sandbox.id)`.

In [ ]:
job = coll.generate_image(
    prompt="A futuristic cityscape at sunset, neon lights reflecting off glass skyscrapers, cyberpunk style",
    model_name=IMAGE_MODEL,
    sandbox_id=sandbox.id,
)

image = job.wait(timeout=900, interval=5)
print(image)

In [ ]:
job = coll.generate_image(
    prompt="A photorealistic portrait of a robot reading a book in a cozy library",
    model_name=IMAGE_MODEL,
    sandbox_id=sandbox.id,
    config={
        "size": "1024x1536",
        "num_inference_steps": 50,
        "guidance_scale": 4.0,
        "negative_prompt": "blurry, low quality, watermark",
    },
)

image = job.wait(timeout=900, interval=5)
print(image)

## 🎞️ 5. Combine FLUX + OmniVoice

Generate an image and narration on the same sandbox, then compose them on a VideoDB timeline.

In [ ]:
image_job = coll.generate_image(
    prompt="A dramatic mountain landscape at dawn, golden hour lighting, cinematic wide shot",
    model_name=IMAGE_MODEL,
    sandbox_id=sandbox.id,
    config={"size": "1280x720", "num_inference_steps": 28},
)
image = image_job.wait(timeout=900, interval=5)
print(f"Image: {image.id}")

audio_job = coll.generate_voice(
    text="From one prompt to a full media moment: generated visuals, synthetic narration, and a playable VideoDB stream.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
    config={"instructions": "female, young adult, moderate pitch, calm and cinematic"},
)
audio = audio_job.wait(timeout=900, interval=5)
print(f"Audio: {audio.id}, length: {audio.length}s")

In [ ]:
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#000000"

image_track = Track()
image_track.add_clip(0, Clip(asset=ImageAsset(id=image.id), duration=float(audio.length), fit=Fit.crop))

audio_track = Track()
audio_track.add_clip(0, Clip(asset=AudioAsset(id=audio.id), duration=float(audio.length)))

timeline.add_track(image_track)
timeline.add_track(audio_track)

stream_url = timeline.generate_stream()
player_url = f"https://player.videodb.io/watch?v={stream_url}"
print(f"Stream: {stream_url}")
print(f"Player: {player_url}")

## 🛑 6. Stop the sandbox

Sandbox billing is based on runtime. Stop the sandbox when finished.

In [ ]:
sandbox.stop()
print(f"Sandbox {sandbox.id} status: {sandbox.status}")

sandbox.wait_for_stop(timeout=120, interval=5)
print(f"Sandbox {sandbox.id} final status: {sandbox.status}")